# Immobilier et performance énergétique en France

Projet du cours Python pour la data science, ENSAE Paris.
Marc Olivier Bouiti Banza, Karlane Vasthi Foudjin Sipeuwo, Aniela Nomwendé Tania Zoungrana.

Ce notebook est le rapport du projet. Il s'exécute du début à la fin à partir des fichiers produits par la chaîne de traitement (voir le README).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

## Introduction

Au moment d'acheter un logement, les dépenses d'énergie et les travaux à prévoir peuvent compter dans le choix du bien et dans le prix que l'acquéreur est prêt à payer. Le diagnostic de performance énergétique (DPE) lui donne un repère en classant les logements de A à G selon leur consommation conventionnelle d'énergie et leurs émissions de gaz à effet de serre. Les classes F et G correspondent aux logements les moins performants, couramment appelés « passoires énergétiques ». Si cette information entre dans les décisions d'achat, elle peut aussi se retrouver dans les prix de vente.

Pour les biens destinés à la location, le classement énergétique a également des conséquences réglementaires. Le DPE, réformé et devenu opposable en juillet 2021, sert de référence aux mesures prévues par la loi Climat et résilience. En métropole, le gel des loyers des logements F et G du parc privé s'applique depuis août 2022 aux baux concernés. Depuis janvier 2025, les logements G ne remplissent plus le critère de décence énergétique exigé pour les nouveaux baux d'habitation, les renouvellements et les reconductions tacites (Ministère chargé du logement, 2025a, 2025b). Ces contraintes peuvent amener les acquéreurs à anticiper des travaux ou des revenus locatifs différents. La période 2022–2025 permet ainsi d'étudier les écarts de prix alors que ces mesures entrent progressivement en application.

Une comparaison entre étiquettes ne suffit pourtant pas à comprendre ces écarts. Les logements les mieux classés peuvent aussi être plus récents, plus grands ou situés dans des communes plus chères. Sur les ventes françaises de 2016 à 2021, Chareyron (2026) observe que les logements mieux classés se vendent plus cher, après prise en compte de plusieurs caractéristiques des biens. Notre projet reprend cette question sur une période plus récente : les logements mieux classés se vendent-ils plus cher au m² lorsque l'on compare des biens semblables, et ces écarts ont-ils évolué entre 2022 et 2025, notamment pour les classes F et G ?

Nous rapprochons pour cela les ventes enregistrées dans les Demandes de valeurs foncières (DVF) de la DGFiP et les diagnostics des logements existants diffusés par l'ADEME depuis juillet 2021. Ce rapprochement permet d'associer un prix de vente à une étiquette énergétique. Après nettoyage, appariement et retrait des ventes rapprochées, nous disposons de 1 568 214 ventes de maisons et d'appartements dans 93 départements métropolitains. Nous décrivons d'abord les biens vendus et les écarts bruts de prix au m², puis nous comparons les ventes au sein de groupes définis par la commune, le type de bien, la tranche de surface, la période de construction et le trimestre de vente. L'évolution de ces écarts est ensuite examinée année par année.

La démarche reste descriptive : même au sein de ces groupes, certaines caractéristiques des logements ne sont pas observées et l'appariement peut comporter des erreurs. Une évolution des écarts de prix ne pourra donc pas être attribuée à elle seule aux changements réglementaires. Le rapport présente d'abord la construction des données et de l'échantillon, puis les résultats des comparaisons. Il se termine par une vérification fondée sur un appariement plus strict et par une discussion des limites de l'analyse.

## Partie 1. Les données

### 1.1 Sources et traitement

*À rédiger.* Les trois sources, la chaîne de traitement et les six règles d'appariement (détail en annexe A).

![Chaîne de traitement](../docs/chaine_de_traitement.png)

In [ ]:
# Seules les colonnes utilisées dans le rapport sont chargées, pour limiter la mémoire nécessaire.
COLONNES_RAPPORT = [
    "date_mutation", "type_bien", "surface", "nb_pieces", "prix_m2", "code_commune", "code_departement",
    "apparie", "vente_rapprochee", "methode_appariement", "ambigu",
    "etiquette_dpe", "conso_5_usages_par_m2_ep", "periode_construction", "annee_construction",
]
ventes = chargement.charger_ventes(colonnes=COLONNES_RAPPORT)
echantillon = chargement.echantillon_analyse(ventes)
print(f"{len(ventes):,} ventes ; {ventes['apparie'].sum():,} appariées ; "
      f"{len(echantillon):,} dans l'échantillon d'analyse")

### 1.2 Le marché

Prix médian au m² de toutes les ventes, appariées ou non, par trimestre et par département.

In [ ]:
prix_trimestre = (ventes.groupby(["trimestre", "type_bien"], observed=True)["prix_m2"]
                  .median()
                  .unstack())

ax = prix_trimestre.plot(marker="o", color={"appartement": "#14213D", "maison": "#A23B3B"})
ax.set_xticks(range(len(prix_trimestre)), prix_trimestre.index, rotation=45, ha="right")
ax.set(title="Prix médian au m² par trimestre (toutes les ventes)", xlabel="", ylabel="€/m²")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
from matplotlib.colors import ListedColormap

# Contours des départements (projet france-geojson de Grégoire David)
URL_DEPARTEMENTS = ("https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
                    "master/departements-version-simplifiee.geojson")
departements = gpd.read_file(URL_DEPARTEMENTS)

prix_dep = ventes.groupby("code_departement", observed=True)["prix_m2"].median().reset_index()
prix_dep["code_departement"] = prix_dep["code_departement"].astype(str)
carte = departements.merge(prix_dep, left_on="code", right_on="code_departement", how="left")

# Classes de prix : sinon Paris (plus de 10 000 €/m²) écraserait toutes les autres couleurs
CLASSES = [0, 1500, 2000, 2500, 3000, 4000, 20000]
NOMS = ["moins de 1 500", "1 500 à 2 000", "2 000 à 2 500", "2 500 à 3 000", "3 000 à 4 000", "plus de 4 000"]
carte["classe_prix"] = pd.cut(carte["prix_m2"], CLASSES, labels=NOMS)

bleus = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS))))
fig, ax = plt.subplots(figsize=(7, 7))
carte.plot(column="classe_prix", cmap=bleus, categorical=True, legend=True, ax=ax,
           edgecolor="white", linewidth=0.3,
           missing_kwds={"color": "lightgrey", "label": "hors champ"},
           legend_kwds={"title": "€/m²", "loc": "center left", "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Prix médian au m² par département, 2022 à 2025")
ax.set_axis_off()
plt.show()

En gris : la Moselle, le Bas-Rhin et le Haut-Rhin, absents de la base des ventes (régime du livre foncier).

*À rédiger.* Commentaire des deux graphiques.

### 1.3 L'échantillon

Part des ventes auxquelles un DPE a été rattaché, puis comparaison des ventes appariées et non appariées. Le détail par département est en annexe B.

In [ ]:
taux = (ventes.groupby(["annee", "type_bien"], observed=True)["apparie"]
        .mean()
        .mul(100)
        .unstack())
taux["ensemble"] = ventes.groupby("annee")["apparie"].mean() * 100
taux.round(1)

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison = (ventes.groupby(["annee", statut], observed=True)
               .agg(ventes=("prix_m2", "size"),
                    prix_m2_median=("prix_m2", "median"),
                    surface_mediane=("surface", "median"),
                    part_maisons=("type_bien", lambda x: (x == "maison").mean() * 100))
               .round(1))
comparaison.index.names = ["année", "statut"]
comparaison

*À rédiger.* Commentaire : évolution du taux d'appariement et différences entre ventes appariées et non appariées.

## Partie 2. Les résultats

Toutes les analyses de cette partie portent sur l'échantillon d'analyse : les ventes appariées, sans les ventes rapprochées.

### 2.1 Description par étiquette

L'analyse porte sur 1 568 214 ventes réalisées entre 2022 et 2025 dans 93 départements métropolitains. Nous retenons les ventes appariées à un DPE, après exclusion des ventes rapprochées. Dans les sections 2.1 et 2.2, le prix étudié est toujours le prix de vente au m², exprimé en €/m² ; la surface est exprimée en m². Les résultats décrivent les biens vendus dans cet échantillon et ne peuvent pas être directement étendus à l'ensemble du parc de logements.

**Caractéristiques des biens vendus**

In [ ]:
from src import cartographie, graphiques

# Description des ventes de chaque classe DPE
description = (
    echantillon.groupby("etiquette_dpe", observed=True)
    .agg(
        nombre_ventes=("prix_m2", "size"),
        prix_m2_moyen=("prix_m2", "mean"),
        prix_m2_median=("prix_m2", "median"),
        surface_mediane=("surface", "median"),
    )
)

par_etiquette = echantillon.groupby("etiquette_dpe", observed=True)
description["maisons_pct"] = par_etiquette["type_bien"].apply(lambda x: (x == "maison").mean() * 100)
description["avant_1975_pct"] = par_etiquette["periode_construction"].apply(
    lambda x: x.isin(["avant 1948", "1948-1974"]).mean() * 100)
description["depuis_2013_pct"] = par_etiquette["periode_construction"].apply(
    lambda x: x.isin(["2013-2021", "après 2021"]).mean() * 100)

description.round(2)

*Lecture : les pourcentages sont calculés au sein de chaque étiquette. Les deux dernières colonnes reposent sur les périodes de construction déclarées : « avant 1948 » et « 1948-1974 » pour la première ; « 2013-2021 » et « après 2021 » pour la seconde. La mention « < 0,1 » désigne une part positive inférieure à 0,1 %.  Les arrondis concernent uniquement la présentation.*

Les classes C et D regroupent à elles seules 61,8 % des ventes. Derrière cette répartition, les biens vendus présentent des profils assez différents : les maisons représentent 83,1 % des ventes en A, contre 38,8 % en C. La surface médiane atteint 104 m² en A, contre 65,82 m² en G. Les écarts de prix au m² entre étiquettes s'observent donc entre des groupes dont la composition diffère.

Le contraste apparaît aussi dans les périodes de construction. Parmi les ventes classées A, 77,9 % concernent des biens construits à partir de 2013, d'après la période déclarée. À l'autre extrémité, les constructions antérieures à 1975 représentent 93,3 % des ventes en F et 97,2 % en G. Plus de la moitié des biens vendus en G, soit 54,7 %, datent même d'avant 1948. La répartition détaillée par période figure dans le tableau ci-dessous. Ces informations restent distinctes de l'année précise de construction, qui n'est pas toujours exploitable pour les corrélations.

In [ ]:
ORDRE_PERIODES = ["avant 1948", "1948-1974", "1975-1977", "1978-1982", "1983-1988",
                  "1989-2000", "2001-2005", "2006-2012", "2013-2021", "après 2021"]

# Répartition des périodes de construction à l'intérieur de chaque étiquette (en %)
(pd.crosstab(echantillon["periode_construction"], echantillon["etiquette_dpe"], normalize="columns")
   .reindex(ORDRE_PERIODES)
   .mul(100)
   .round(1))

**Répartition géographique des ventes de logements F ou G**

In [ ]:
# Part des ventes de logements F ou G par département
tableau_fg = cartographie.parts_fg(echantillon)
fond = cartographie.fond_departements(RACINE / "data" / "external")

fig = cartographie.carte_fg(
    tableau_fg,
    fond,
    titre="Ventes de logements F ou G par département, 2022 à 2025",
)
fig.savefig(config.REPORTS / "figures" / "carte_part_fg_france_noms.png", dpi=250, bbox_inches="tight")
plt.show()

total_ventes = int(tableau_fg["nombre_ventes"].sum())
total_fg = int(tableau_fg["nombre_fg"].sum())
print(f"Ventes F ou G : {total_fg:,} sur {total_ventes:,}, soit {100 * total_fg / total_ventes:.2f} %")

La carte rapporte, pour chaque département, le nombre de ventes retenues de logements F ou G au nombre total de ventes retenues. Sur l'ensemble de l'échantillon, ces deux classes représentent 196 246 ventes sur 1 568 214, soit 12,51 %. Cette proportion est calculée à partir des effectifs totaux, sans faire la moyenne simple des pourcentages départementaux.

Les différences géographiques sont marquées. La part F/G atteint 40,04 % dans la Creuse (645 ventes sur 1 611), 30,64 % dans le Cantal (790 sur 2 578) et 28,26 % dans la Nièvre (1 255 sur 4 441). Elle est plus faible dans l'Aude, à 2,01 % (213 sur 10 614), en Haute-Corse, à 2,47 % (30 sur 1 216), et dans l'Hérault, à 2,56 % (1 151 sur 44 989).

La Moselle, le Bas-Rhin et le Haut-Rhin apparaissent en gris car aucune vente de l'échantillon n'y est observée. Il ne faut donc pas lire cette couleur comme une absence de logements F ou G. Les territoires ultramarins ne sont pas représentés sur cette carte métropolitaine. Enfin, les écarts entre départements portent sur les ventes retenues : ils peuvent refléter les caractéristiques des biens vendus et la sélection liée à l'appariement, sans que la carte permette de distinguer ces explications.

*Sources : données DVF–DPE préparées du projet ; fond départemental Etalab, millésime 2025, généralisation 100 m, projection Lambert-93.*

**Corrélations de Spearman**

In [ ]:
ORDRE_PERIODES = ["avant 1948", "1948-1974", "1975-1977", "1978-1982", "1983-1988",
                  "1989-2000", "2001-2005", "2006-2012", "2013-2021", "après 2021"]

# Année de construction : retenue seulement si elle est entière, positive, antérieure à la vente
# et compatible avec la période déclarée
annee = pd.to_numeric(echantillon["annee_construction"], errors="coerce")
annee_vente = echantillon["date_mutation"].dt.year
bornes = [float("-inf"), 1947, 1974, 1977, 1982, 1988, 2000, 2005, 2012, 2021, float("inf")]
periode_calculee = pd.cut(annee, bins=bornes, labels=ORDRE_PERIODES)
coherence = (periode_calculee.astype("string")
             .eq(echantillon["periode_construction"].astype("string").str.strip())
             .fillna(False))
annee_valide = annee.gt(0) & annee.mod(1).eq(0) & annee.le(annee_vente) & coherence

controle_annee = pd.Series({
    "Année absente ou non numérique": int(annee.isna().sum()),
    "Année renseignée non retenue": int((annee.notna() & ~annee_valide).sum()),
    "Année utilisable après contrôles": int(annee_valide.sum()),
}, name="nombre_ventes")

# Tableau des variables, avec l'étiquette codée de 1 (A) à 7 (G)
donnees_correlation = echantillon[["prix_m2", "surface", "conso_5_usages_par_m2_ep"]].apply(
    pd.to_numeric, errors="coerce")
donnees_correlation["rang_dpe"] = echantillon["etiquette_dpe"].map(
    {lettre: rang for rang, lettre in enumerate("ABCDEFG", start=1)}).astype(float)
donnees_correlation["annee_construction"] = annee.where(annee_valide)

correlations = donnees_correlation.corr(method="spearman")
presences = donnees_correlation.notna().astype("int64")
effectifs_correlations = presences.T.dot(presences)

controle_annee.to_frame()

In [ ]:
fig = graphiques.matrice_spearman(
    correlations,
    effectifs_correlations,
    titre="Corrélations de Spearman, 93 départements, 2022 à 2025",
)
fig.savefig(config.REPORTS / "figures" / "correlations_spearman_france.png", dpi=250, bbox_inches="tight")
plt.show()

La matrice met en relation le prix au m², la surface, la consommation conventionnelle d'énergie primaire par m² et par an, le rang DPE et l'année de construction. L'étiquette est codée de 1 pour A à 7 pour G : plus le rang est élevé, moins le classement énergétique est favorable.

La consommation et le rang DPE sont fortement associés, avec une corrélation de 0,921. L'année de construction est, quant à elle, négativement corrélée à la consommation (−0,568) et au rang DPE (−0,563). Parmi les observations disposant d'une année exploitable, les logements plus récents tendent ainsi à être moins consommateurs et mieux classés.

Les liens avec le prix au m² sont moins prononcés. Sa corrélation est de −0,132 avec le rang DPE et de −0,095 avec la consommation. Elle atteint −0,284 avec la surface et +0,165 avec l'année de construction. Ces coefficients décrivent des associations entre les rangs des observations ; ils ne s'interprètent ni comme des variations de prix en pourcentage, ni comme des effets causaux.

Les effectifs varient selon les variables disponibles. Les corrélations entre prix au m², surface et rang DPE reposent sur 1 568 214 ventes, contre 1 568 087 lorsque la consommation intervient. Chaque calcul utilise uniquement les ventes renseignées pour les deux variables concernées.

Pour l'année de construction, 598 081 valeurs sont absentes ou non numériques et 32 535 années renseignées sont écartées après vérification. Une année est retenue si elle est entière, positive, non postérieure à la vente et compatible avec la période de construction déclarée. Il reste 937 598 années exploitables, dont 937 496 avec une consommation renseignée. Aucune année n'est imputée. Les ventes concernées restent utilisées pour les autres corrélations lorsque leurs données le permettent. Les résultats faisant intervenir l'année doivent donc être lus en gardant à l'esprit qu'ils portent sur un sous-échantillon.

### 2.2 Comparaison sans correction

Pour examiner les écarts de prix au m² entre étiquettes, nous distinguons les maisons des appartements. Les boîtes à moustaches présentent les distributions de chaque groupe sur une même échelle verticale, ce qui permet de comparer à la fois leurs médianes et leur dispersion.

In [ ]:
fig = graphiques.boxplots_prix_dpe(
    echantillon,
    titre="Prix au m² selon le DPE, 93 départements, 2022 à 2025",
)
fig.savefig(config.REPORTS / "figures" / "boxplots_dpe_france.png", dpi=200, bbox_inches="tight")
plt.show()

Chaque boîte contient les 50 % centraux des prix au m², et le trait intérieur indique la médiane. Les moustaches s'étendent jusqu'aux valeurs observées dans la limite de 1,5 fois l'écart interquartile en dessous du premier quartile et au-dessus du troisième. Les points situés au-delà sont masqués pour faciliter la lecture, mais toutes les valeurs renseignées restent incluses dans les calculs.

In [ ]:
medianes_type = (
    echantillon.groupby(["etiquette_dpe", "type_bien"], observed=True)["prix_m2"]
    .median()
    .unstack()
    .rename(columns={"appartement": "prix médian des appartements (€/m²)",
                     "maison": "prix médian des maisons (€/m²)"})
)
medianes_type.round(2)

Pour les maisons, le prix médian au m² diminue à mesure que l'on passe de A à G : il va de 3 055,68 à 1 760,00 €/m². La médiane en G est ainsi inférieure de 42,4 % à celle de A. Les distributions se recouvrent néanmoins largement : l'étiquette seule ne suffit pas à caractériser le prix au m² d'une vente.

Le profil des appartements est différent. Leur prix médian au m² baisse de A à D, puis remonte de E à G. Il atteint 4 998,09 €/m² en G, soit 19,9 % de plus qu'en A. Cette comparaison concerne toutefois des biens de tailles différentes : la surface médiane des appartements G est d'environ 34,2 m², contre 61,2 m² en A. La hausse observée pour les classes les moins favorables ne permet donc pas de conclure qu'un mauvais DPE augmenterait le prix d'un appartement.

Les deux écarts entre G et A sont calculés à partir des médianes non arrondies, selon la formule `100 × (médiane G / médiane A − 1)`, séparément pour chaque type de bien. Ils restent des écarts bruts entre groupes. Même après avoir séparé maisons et appartements, des différences de localisation, de surface, de période de construction et de date de vente subsistent. La section 2.3 prolonge cette analyse en comparant des logements semblables.

### 2.3 Résultat principal : l'écart à logement comparable

Cette partie présente les différences de prix au mètre carré observées entre les logements selon leur étiquette énergétique. Deux références sont utilisées : la classe G, qui correspond à la moins bonne performance énergétique, et la classe D, située au milieu de l'échelle.

Il faut distinguer deux mesures. Le premier écart compare directement les prix moyens des logements de deux classes. Il ne tient donc pas compte des différences de composition entre les groupes. Le second écart est calculé sur des groupes de logements comparables, constitués à partir de la commune, du type de bien, de la tranche de surface, de la période de construction et du trimestre de vente. Cette seconde mesure permet de limiter l'influence de ces caractéristiques sur la comparaison.

Les intervalles de confiance à 95 % sont obtenus par bootstrap, en rééchantillonnant les communes. Cette méthode tient compte du fait que les observations d'une même commune peuvent être liées entre elles. Les estimations présentées reposent sur 1 000 répétitions.

In [ ]:
from src import produire_resultats, produire_robustesse, produire_evolution_annuelle
from src import visualiser_resultats, visualiser_evolution_annuelle

# Écarts bruts et comparables, références G et D, intervalles par bootstrap (1 000 répétitions)
resultats = produire_resultats.calculer_resultats(ventes)


def tableau_ecarts(resultats, reference):
    t = resultats[resultats["reference"] == reference].set_index("etiquette")
    return pd.DataFrame({
        "écart brut (%)": t["ecart_brut_pct"],
        "écart à logement comparable (%)": t["ecart_comparable_pct"],
        "IC 95 % bas": t["ic_comparable_inf"],
        "IC 95 % haut": t["ic_comparable_sup"],
        "part de la classe couverte (%)": t["part_cible_support_pct"],
    }).round(2)

#### 2.3.1 Comparaison avec la classe G

In [ ]:
tableau_ecarts(resultats, "G")

Les écarts sont positifs pour toutes les classes comparées à G. Autrement dit, dans notre échantillon, les logements ayant une meilleure étiquette énergétique se vendent en moyenne plus cher que les logements classés G. L'écart est particulièrement marqué pour les classes A, B et C.

La comparaison entre logements similaires donne également des écarts positifs. Elle montre notamment que la différence entre F et G, peu visible dans la comparaison brute, atteint 2,54 % après prise en compte des caractéristiques retenues pour former les groupes comparables.

Il faut toutefois rester prudent pour les classes A et B. Seules 0,42 % des ventes classées A et 2,04 % des ventes classées B se trouvent dans un groupe qui contient aussi des logements classés G (colonne « part de la classe couverte »). Ces comparaisons reposent donc sur peu de ventes et sont plus sensibles à la composition de l'échantillon.

#### 2.3.2 Comparaison avec la classe D

In [ ]:
tableau_ecarts(resultats, "D")

La classe D permet d'observer les écarts de part et d'autre d'une étiquette intermédiaire. Les classes A, B et C présentent des prix plus élevés que D, tandis que les classes E, F et G présentent des prix plus faibles.

Après comparaison de logements similaires, l'écart atteint -8,72 % pour F et -11,00 % pour G par rapport à D. Les intervalles de confiance correspondants ne contiennent pas zéro. Dans le cadre de cette méthode, les écarts estimés sont donc relativement précis.

La comparaison brute et la comparaison entre logements similaires ne donnent pas toujours le même ordre de grandeur. C'est notamment le cas de la classe C, pour laquelle l'écart avec D passe de 10,54 % à 3,94 %. Cela montre l'importance de tenir compte des caractéristiques des logements et de leur localisation avant d'interpréter les différences de prix.

In [ ]:
fig = visualiser_resultats.graphique_ecarts(resultats)
plt.show()

Dans l'ensemble, les résultats font apparaître une association entre la performance énergétique et le prix de vente. Ils ne permettent cependant pas d'affirmer que l'amélioration de l'étiquette énergétique entraîne, à elle seule, une hausse du prix. D'autres caractéristiques, qui ne sont pas toutes observées dans les données, peuvent également expliquer une partie des différences.

### 2.4 Évolution des écarts de 2022 à 2025

Les résultats précédents portent sur l'ensemble de la période étudiée. Il est également intéressant de vérifier si les écarts entre les logements classés F ou G et ceux classés D évoluent d'une année à l'autre.

Pour cette analyse, la classe D est conservée comme référence. Les écarts sont calculés séparément pour chaque année, selon la même logique de comparaison entre logements similaires.

In [ ]:
evolution = produire_evolution_annuelle.calculer_evolution(ventes)
evolution[["annee", "etiquette", "ecart_brut_pct", "ecart_comparable_pct", "part_cible_support_pct"]].round(2)

In [ ]:
fig = visualiser_evolution_annuelle.graphique_evolution(evolution)
plt.show()

Les résultats montrent que les logements classés F et G se vendent moins cher que ceux de classe D sur chacune des années observées. L'écart comparable devient globalement plus important au cours de la période.

Pour les logements classés F, il passe de -6,72 % en 2022 à -10,54 % en 2025. Pour les logements classés G, il évolue de -8,07 % à -13,18 %. Cette évolution est particulièrement visible pour la classe G.

Cette période correspond également à plusieurs changements réglementaires concernant les logements les moins performants sur le plan énergétique :

- **24 août 2022 :** entrée en vigueur du gel des augmentations de loyers pour les logements classés F et G en France métropolitaine, pour les contrats concernés.
- **1er janvier 2023 :** le seuil de 450 kWh d'énergie finale par mètre carré et par an entre dans le critère de décence énergétique. Les logements qui dépassent ce seuil ne satisfont plus ce critère.
- **1er janvier 2025 :** les logements classés G sont concernés par l'interdiction de location au titre du calendrier de décence énergétique.

Ces dates sont reportées sur le graphique afin de situer l'évolution des écarts dans leur contexte réglementaire. Elles ne permettent toutefois pas d'attribuer directement les variations observées à ces mesures. D'autres facteurs ont pu influencer le marché immobilier pendant la même période.

Enfin, la part de logements de la classe cible couverte par les comparaisons augmente au fil des années. Les résultats annuels doivent donc être interprétés en tenant compte de cette évolution de la couverture. Les intervalles de confiance présentés dans la section précédente ne sont pas recalculés ici pour chaque année.

## Partie 3. Vérification et limites

### 3.1 Appariement strict

Les résultats peuvent dépendre de la fiabilité de l'appariement entre les ventes et les diagnostics. Pour vérifier ce point, une seconde analyse ne garde que les appariements les plus sûrs : elle retire les ventes marquées comme ambiguës et celles appariées par la distance plutôt que par l'adresse.

Les estimations obtenues avec cette méthode sont comparées à celles de l'analyse principale. Les intervalles de confiance sont également présentés pour les deux méthodes.

In [ ]:
resultats_stricts = produire_resultats.calculer_resultats(ventes, strict=True)
comparaison_strict = produire_robustesse.comparer_standard_strict(resultats, resultats_stricts)
comparaison_strict[["reference", "etiquette",
                    "ecart_comparable_pct_standard", "ic_comparable_inf_standard", "ic_comparable_sup_standard",
                    "ecart_comparable_pct_strict", "ic_comparable_inf_strict", "ic_comparable_sup_strict"]].round(2)

Les estimations obtenues avec l'appariement strict sont légèrement plus élevées en valeur absolue que celles de l'analyse principale. C'est le cas, par exemple, de l'écart entre les classes G et D, qui passe de -11,00 % à -11,70 %.

Malgré ces différences, les résultats conservent le même sens : les logements mieux classés présentent des prix plus élevés que ceux des classes énergétiques moins performantes, dans les comparaisons effectuées. Les intervalles de confiance restent également du même côté de zéro.

Cette vérification est plutôt rassurante : les principaux constats ne reposent pas uniquement sur le choix des critères d'appariement de l'analyse initiale. Il faut néanmoins noter que les critères plus stricts réduisent le nombre de logements pouvant être comparés. La couverture de l'échantillon est donc un peu plus faible.

### 3.2 Limites de l'étude

Plusieurs limites doivent être prises en compte pour interpréter les résultats.

**La qualité de l'appariement.** Le rapprochement entre les données de ventes et les diagnostics énergétiques repose sur des informations de localisation et sur les caractéristiques des biens. Malgré les contrôles effectués, certaines correspondances peuvent rester incertaines. Ce problème peut notamment se poser dans les immeubles comportant plusieurs logements à une même adresse.

**La sélection des logements observés.** L'étude porte sur des logements vendus et pour lesquels un diagnostic énergétique a pu être associé. Elle ne couvre donc pas l'ensemble du parc immobilier français. Les logements qui ne sont pas vendus pendant la période étudiée ne figurent pas dans l'échantillon.

**Les caractéristiques non observées.** La comparaison tient compte de plusieurs éléments, comme la commune, le type de bien, la surface, la période de construction et le trimestre de vente. D'autres facteurs peuvent néanmoins influencer le prix : l'état général du logement, sa localisation précise, la présence d'un extérieur, l'étage, la qualité des équipements ou encore les travaux réalisés.

**La représentation inégale des classes énergétiques.** Certaines classes, en particulier A et B dans la comparaison avec G, sont peu représentées. Les résultats qui les concernent doivent donc être lus avec davantage de prudence.

**L'interprétation des écarts.** Les méthodes utilisées permettent de comparer les prix observés entre des groupes de logements présentant des caractéristiques proches. Elles ne suffisent pas à établir un lien de causalité. Il n'est donc pas possible d'affirmer que la seule amélioration du DPE entraîne la hausse du prix de vente.

Enfin, le bootstrap par commune permet de tenir compte de la dépendance possible entre les observations d'une même commune pour le calcul des intervalles de confiance. Il ne corrige cependant pas les autres limites liées à la qualité des données ou aux caractéristiques qui ne sont pas observées.

## Conclusion

Cette étude avait pour objectif d'examiner le lien entre la performance énergétique des logements et leur prix de vente en France. Pour cela, les données de ventes immobilières ont été rapprochées des diagnostics de performance énergétique, puis les prix ont été comparés selon les différentes étiquettes du DPE.

Les résultats font apparaître des écarts de prix selon la classe énergétique. Dans la comparaison entre logements similaires, les biens classés F se vendent en moyenne 8,72 % moins cher que ceux classés D, tandis que l'écart atteint 11,00 % pour les logements classés G. Les comparaisons avec la classe G montrent également des écarts positifs pour les classes énergétiques supérieures.

L'analyse annuelle indique que les écarts entre les classes F et G et la classe D deviennent globalement plus importants entre 2022 et 2025. La vérification avec des critères d'appariement plus stricts conduit à des résultats proches de ceux de l'analyse principale. Les principaux constats semblent donc relativement stables face à cette modification de la méthode.

Ces résultats doivent toutefois être interprétés avec prudence. Les données disponibles ne permettent pas de tenir compte de toutes les caractéristiques susceptibles d'influencer le prix d'un logement. De plus, l'étude repose sur les biens vendus et correctement rapprochés avec un diagnostic énergétique. Elle met donc en évidence une association entre l'étiquette énergétique et le prix, mais ne permet pas de conclure à un effet causal du DPE sur la valeur immobilière.

Malgré ces limites, les résultats suggèrent que la performance énergétique constitue un élément à prendre en considération dans l'analyse du marché immobilier. Les écarts observés entre les classes D, F et G sont cohérents avec l'idée que les logements les moins performants peuvent faire l'objet d'une décote sur le marché. Cette question mérite une attention particulière dans un contexte où les exigences réglementaires relatives à la location des logements énergivores se renforcent.



## Annexes

### Annexe A. Règles d'appariement

Une vente est rattachée à un DPE si les six règles suivantes sont respectées. Les seuils sont définis dans `src/config.py`.

| Règle | Contenu | Seuil |
|---|---|---|
| 1. Même adresse et même type de logement | Même commune, même voie et même numéro, après avoir écrit les adresses des deux bases de la même façon | |
| 2. À défaut, un DPE tout proche | Quand l'adresse ne permet pas de conclure, DPE situé à faible distance de la vente | 20 m pour un appartement, 60 m pour une maison |
| 3. Une surface proche | Écart entre la surface de la vente (DVF) et la surface habitable du DPE | au plus 10 % pour un appartement, 20 % pour une maison |
| 4. Un DPE établi avant la vente | DPE antérieur à la vente ; son ancienneté entre dans le score, si bien qu'à surface égale le plus récent est préféré | au plus 3 ans avant la vente |
| 5. Un DPE pour un seul logement | Un même DPE ne peut pas être attribué à deux logements différents ; il revient à celui qui lui correspond le mieux. Plusieurs ventes d'un même logement peuvent en revanche partager son DPE | |
| 6. Les cas ambigus sont signalés | Vente marquée « ambiguë » quand un autre DPE d'étiquette différente convient presque aussi bien | écart de score inférieur à 0,10 |

Quand plusieurs DPE respectent ces règles, on retient celui dont le score est le plus faible :

score = écart de surface / tolérance + 0,25 × ancienneté du DPE / 3 ans + distance / rayon (la distance ne compte que pour la règle 2).

Sont aussi écartés : les DPE dont l'adresse a été mal reconnue par la Base Adresse Nationale (note de fiabilité inférieure à 0,5), les DPE remplacés par un DPE plus récent et les DPE qui ne portent pas sur une maison ou un appartement (DPE d'immeuble entier). Les ventes d'un même logement à moins de 30 jours d'écart sont signalées et exclues de l'échantillon d'analyse.

### Annexe B. Taux d'appariement par département

Le taux par année et par type de bien est donné en partie 1.3.

In [ ]:
taux_dep = (ventes.groupby("code_departement", observed=True)["apparie"]
            .mean()
            .mul(100)
            .reset_index(name="taux"))
taux_dep["code_departement"] = taux_dep["code_departement"].astype(str)
carte_taux = departements.merge(taux_dep, left_on="code", right_on="code_departement", how="left")

CLASSES_TAUX = [0, 30, 40, 50, 60, 100]
NOMS_TAUX = ["moins de 30 %", "30 à 40 %", "40 à 50 %", "50 à 60 %", "plus de 60 %"]
carte_taux["classe_taux"] = pd.cut(carte_taux["taux"], CLASSES_TAUX, labels=NOMS_TAUX)

bleus_taux = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS_TAUX))))
fig, ax = plt.subplots(figsize=(7, 7))
carte_taux.plot(column="classe_taux", cmap=bleus_taux, categorical=True, legend=True, ax=ax,
                edgecolor="white", linewidth=0.3,
                missing_kwds={"color": "lightgrey", "label": "hors champ"},
                legend_kwds={"title": "Ventes appariées", "loc": "center left",
                             "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Part des ventes appariées à un DPE, 2022 à 2025")
ax.set_axis_off()
plt.show()

In [ ]:
taux_dep_annee = (ventes.groupby(["code_departement", "annee"], observed=True)["apparie"]
                  .mean()
                  .mul(100)
                  .unstack()
                  .round(1))
taux_dep_annee["ensemble"] = ventes.groupby("code_departement", observed=True)["apparie"].mean().mul(100).round(1)
taux_dep_annee.sort_values("ensemble")

### Annexe C. Ventes appariées et non appariées

Comparaison sur les variables connues pour toutes les ventes, par type de bien, sur l'ensemble de la période.

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison_detail = (ventes.groupby(["type_bien", statut], observed=True)
                      .agg(ventes=("prix_m2", "size"),
                           prix_m2_q1=("prix_m2", lambda x: x.quantile(0.25)),
                           prix_m2_median=("prix_m2", "median"),
                           prix_m2_q3=("prix_m2", lambda x: x.quantile(0.75)),
                           surface_mediane=("surface", "median"),
                           pieces_mediane=("nb_pieces", "median"))
                      .round(1))
comparaison_detail.index.names = ["type de bien", "statut"]
comparaison_detail

Répartition des étiquettes dans l'échantillon d'analyse, comparée à celle des résidences principales au 1er janvier 2024 (SDES, 2024). Le parc ne tient pas compte du changement des seuils pour les logements de moins de 40 m² entré en vigueur le 1er juillet 2024.

In [ ]:
# Résidences principales au 1er janvier 2024 (SDES, 2024), en %
PARC_2024 = pd.Series({"A": 2.6, "B": 4.6, "C": 25.7, "D": 32.1, "E": 21.1, "F": 8.1, "G": 5.7})

repartition = pd.DataFrame({
    "échantillon": echantillon["etiquette_dpe"].value_counts(normalize=True).reindex(ORDRE) * 100,
    "parc 2024": PARC_2024,
}).round(1)
display(repartition)

x = np.arange(len(ORDRE))
largeur = 0.4
fig, ax = plt.subplots()
ax.bar(x - largeur / 2, repartition["échantillon"], largeur, color=[COULEURS[e] for e in ORDRE], label="ventes appariées")
ax.bar(x + largeur / 2, repartition["parc 2024"], largeur, color="lightgrey", label="résidences principales (2024)")
ax.set_xticks(x, ORDRE)
ax.set(ylabel="%", title="Répartition des étiquettes : ventes appariées et parc de logements")
ax.legend(frameon=False)
plt.show()

### Annexe D. Dictionnaire des variables

Variables chargées par `chargement.charger_ventes()`. Une ligne correspond à une vente.

**Vente (source : DVF)**

| Variable | Description |
|---|---|
| `id_mutation` | Identifiant de la vente dans DVF |
| `date_mutation` | Date de la vente |
| `annee`, `trimestre` | Année et trimestre de la vente, calculés à partir de la date |
| `prix` | Prix de vente, en euros |
| `type_bien` | Maison ou appartement |
| `surface` | Surface du logement en m² : surface « loi Carrez » pour un appartement quand elle est renseignée, sinon surface bâtie |
| `nb_pieces` | Nombre de pièces principales |
| `prix_m2` | Prix divisé par la surface, en euros par m² |
| `code_commune`, `code_departement` | Code Insee de la commune et code du département |
| `longitude`, `latitude` | Coordonnées du centre de la parcelle cadastrale |

**Appariement (calculé par la chaîne de traitement)**

| Variable | Description |
|---|---|
| `apparie` | Vrai si un DPE a été rattaché à la vente |
| `vente_rapprochee` | Vrai si le même logement a été vendu une autre fois à moins de 30 jours d'écart (exclue de l'échantillon d'analyse) |
| `methode_appariement` | « adresse » (règle 1) ou « spatial » (règle 2, DPE proche) |
| `ambigu` | Vrai si un autre DPE d'étiquette différente convenait presque aussi bien (règle 6) |
| `n_candidats` | Nombre de DPE qui respectaient les règles pour cette vente |
| `delai_jours` | Nombre de jours entre l'établissement du DPE et la vente |
| `ecart_surface` | Écart relatif entre la surface de la vente et la surface habitable du DPE (0,05 = 5 %) |
| `distance_m` | Distance en mètres entre la vente et le DPE, pour les appariements par proximité |

**Diagnostic (source : base des DPE de l'ADEME, renseigné pour les ventes appariées)**

| Variable | Description |
|---|---|
| `etiquette_dpe` | Étiquette énergie, de A (la meilleure) à G |
| `etiquette_ges` | Étiquette climat (émissions de gaz à effet de serre), de A à G |
| `conso_5_usages_par_m2_ep` | Consommation conventionnelle d'énergie primaire, en kWh par m² et par an, pour cinq usages : chauffage, eau chaude, refroidissement, éclairage et auxiliaires |
| `version_dpe` | Version de la méthode de calcul du DPE |
| `periode_construction` | Période de construction du logement |
| `annee_construction` | Année de construction, quand elle est connue |
| `zone_climatique` | Zone climatique réglementaire du logement (H1, H2 ou H3, avec leurs sous-zones) |

## Références

*À compléter à partir de la note de projet.*